In [ ]:
# # ============================================================================
# # CELL 1: Install Required Packages
# # ============================================================================
# !pip install geopandas pandas requests pillow torch torchvision
# !pip install git+https://github.com/openai/CLIP.git

# ============================================================================
# CELL 2: Import Libraries and Set API Key
# ============================================================================
import pandas as pd
import geopandas as gpd
import requests
import time
from pathlib import Path
import urllib.parse
from PIL import Image
import io

# 🔑 PASTE YOUR GOOGLE API KEY HERE
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"

print("✓ Libraries imported")
print(f"✓ API Key set: {GOOGLE_API_KEY[:20]}..." if GOOGLE_API_KEY != "YOUR_API_KEY_HERE" else "⚠️  PASTE YOUR API KEY ABOVE!")

# ============================================================================
# CELL 3: Load Ferguson Data
# ============================================================================

def load_ferguson_parcels():
    """Load Ferguson parcels from YOUR actual data files"""
    print("Loading Ferguson parcel data...")
    
    # Load parcels shapefile
    parcel_path = '/Users/wvianese/Desktop/Ferguson Vacant Housing Tool/Data/STL Map Base/Parcels.shp'
    parcels = gpd.read_file(parcel_path)
    
    # Filter to Ferguson (ZIP 63135)
    ferguson_parcels = parcels.loc[parcels['PROP_ZIP'] == '63135'].copy()
    print(f"  ✓ Loaded {len(ferguson_parcels):,} parcels in Ferguson (ZIP 63135)")
    
    # Load confirmed abandoned list
    csv_path = '/Users/wvianese/Desktop/Ferguson Vacant Housing Tool/Data/fergusonPropertyRestorationProgram.csv'
    ferguson_csv = pd.read_csv(csv_path)
    ferguson_csv.columns = ferguson_csv.columns.str.strip()
    print(f"  ✓ Loaded {len(ferguson_csv):,} confirmed abandoned properties")
    
    # Merge to identify abandoned parcels
    ferguson_parcels = ferguson_parcels.merge(
        ferguson_csv[['PARCEL #']],
        left_on='LOCATOR',
        right_on='PARCEL #',
        how='left'
    )
    
    # Create abandoned flag
    ferguson_parcels['is_abandoned'] = ferguson_parcels['PARCEL #'].notna().astype(int)
    
    # Clean columns
    for col in ['CODE_ENFOR', 'YEARBLT', 'RESQFT', 'TOTASSMT']:
        ferguson_parcels[col] = pd.to_numeric(ferguson_parcels[col], errors='coerce').fillna(0)
    
    # Filter to parcels with valid addresses
    valid_address = ferguson_parcels['PROP_ADD'].notna() & (ferguson_parcels['PROP_ADD'] != '')
    ferguson_parcels = ferguson_parcels[valid_address].copy()
    
    print(f"  ✓ {ferguson_parcels['is_abandoned'].sum():,} confirmed abandoned parcels")
    print(f"  ✓ {len(ferguson_parcels):,} total parcels with valid addresses")
    
    return ferguson_parcels

parcels_df = load_ferguson_parcels()

# ============================================================================
# CELL 4: Google Street View Functions
# ============================================================================

def get_streetview_url(address, api_key, size="600x400", heading=None, pitch=0, fov=90):
    """Generate Google Street View static image URL WITH API KEY"""
    base_url = "https://maps.googleapis.com/maps/api/streetview"
    
    params = {
        'size': size,
        'location': address,
        'pitch': pitch,
        'fov': fov,
        'source': 'outdoor',
        'key': api_key
    }
    
    if heading is not None:
        params['heading'] = heading
    
    return f"{base_url}?{urllib.parse.urlencode(params)}"

def download_streetview_image(address, save_path, api_key, size="600x400"):
    """Download Street View image using API key"""
    url = get_streetview_url(address, api_key, size)
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            if len(response.content) > 5000:
                with open(save_path, 'wb') as f:
                    f.write(response.content)
                return True, "Success", response.content
            else:
                return False, "No Street View imagery available", None
        else:
            return False, f"HTTP {response.status_code}", None
    except Exception as e:
        return False, str(e), None

print("✓ Street View functions defined")

# ============================================================================
# CELL 5: Collect Street View Images
# ============================================================================

def collect_ferguson_streetview_safe(
    parcels_df,
    api_key,
    output_dir='streetview_images',
    sample_size=None,
    prioritize_abandoned=True,
    delay_seconds=1.0
):
    """Safely collect Street View images with cost tracking"""
    
    if api_key == "YOUR_API_KEY_HERE" or not api_key:
        print("❌ ERROR: You must set your Google API key!")
        return None
    
    df = parcels_df.copy()
    
    # Sample if requested
    if sample_size:
        if prioritize_abandoned:
            abandoned = df[df['is_abandoned'] == 1]
            not_abandoned = df[df['is_abandoned'] == 0]
            
            n_abandoned = min(len(abandoned), sample_size // 2)
            n_not_abandoned = sample_size - n_abandoned
            
            sample_df = pd.concat([
                abandoned.sample(n=n_abandoned, random_state=42) if n_abandoned > 0 else abandoned,
                not_abandoned.sample(n=min(n_not_abandoned, len(not_abandoned)), random_state=42)
            ])
            
            df = sample_df
        else:
            df = df.sample(n=min(sample_size, len(df)), random_state=42)
    
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    cost_per_image = 0.007
    estimated_cost = len(df) * cost_per_image
    estimated_time_minutes = (len(df) * delay_seconds) / 60
    
    print("\n" + "=" * 70)
    print("💰 COST & TIME ESTIMATE")
    print("=" * 70)
    print(f"Total parcels to process: {len(df):,}")
    print(f"  • Confirmed abandoned: {df['is_abandoned'].sum()}")
    print(f"  • Not abandoned: {(df['is_abandoned'] == 0).sum()}")
    print(f"\nEstimated cost: ${estimated_cost:.2f}")
    print(f"Your free credits: $300.00")
    print(f"Remaining after: ${300 - estimated_cost:.2f}")
    print(f"\nEstimated time: {estimated_time_minutes:.1f} minutes")
    print("=" * 70)
    
    proceed = input(f"\n⚠️  Proceed with ${estimated_cost:.2f} charge? (yes/no): ")
    if proceed.lower() != 'yes':
        print("❌ Cancelled")
        return None
    
    print("\n🚀 STARTING COLLECTION\n")
    
    results = []
    cost_tracker = 0.0
    
    for idx, row in df.iterrows():
        parcel_id = row['LOCATOR']
        address = row['PROP_ADD']
        is_abandoned = row['is_abandoned']
        
        full_address = f"{address}, Ferguson, MO 63135"
        safe_filename = f"{parcel_id}_{address.replace(' ', '_').replace(',', '').replace('/', '_')}.jpg"
        filepath = output_path / safe_filename
        
        if filepath.exists():
            print(f"⏭️  {parcel_id} | Already exists")
            results.append({
                'parcel_id': parcel_id,
                'address': address,
                'is_abandoned': is_abandoned,
                'success': True,
                'message': 'Already downloaded',
                'filepath': str(filepath),
                'has_imagery': True
            })
            continue
        
        success, message, image_data = download_streetview_image(full_address, filepath, api_key)
        cost_tracker += cost_per_image
        
        results.append({
            'parcel_id': parcel_id,
            'address': address,
            'is_abandoned': is_abandoned,
            'success': success,
            'message': message,
            'filepath': str(filepath) if success else None,
            'has_imagery': success
        })
        
        status = "✓" if success else "✗"
        abandoned_marker = "🏚️" if is_abandoned == 1 else "🏘️"
        print(f"{status} {abandoned_marker} {parcel_id} | {address[:35]:<35} | {message} | ${cost_tracker:.2f}")
        
        time.sleep(delay_seconds)
        
        if len(results) % 50 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(output_path / 'collection_results_temp.csv', index=False)
            print(f"\n💾 Progress saved: {len(results)}/{len(df)}\n")
    
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_path / 'collection_results.csv', index=False)
    
    success_count = results_df['success'].sum()
    print("\n" + "=" * 70)
    print("✅ COLLECTION COMPLETE")
    print("=" * 70)
    print(f"Successfully collected: {success_count}/{len(df)} ({success_count/len(df)*100:.1f}%)")
    print(f"💰 TOTAL COST: ${cost_tracker:.2f}")
    print("=" * 70)
    
    return results_df

# RUN COLLECTION ON ALL PARCELS
collection_results = collect_ferguson_streetview_safe(
    parcels_df,
    api_key=GOOGLE_API_KEY,
    sample_size=None,  # None = ALL parcels
    delay_seconds=1.0
)

# ============================================================================
# CELL 6: Install CLIP
# ============================================================================
!pip install git+https://github.com/openai/CLIP.git

# ============================================================================
# CELL 7: CLIP Analysis Function (3 Categories)
# ============================================================================
import torch
import clip

def analyze_with_clip(image_path):
    """
    Use CLIP to classify property into 3 categories:
    1. Occupied - Well-maintained building
    2. Abandoned - Deteriorated structure
    3. Vacant Lot - Empty land, no building
    """
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model, preprocess = clip.load("ViT-B/32", device=device)
        
        image = Image.open(image_path).convert('RGB')
        image = preprocess(image).unsqueeze(0).to(device)
        
        text_prompts = [
            "a well-maintained occupied residential house with clear signs of habitation and upkeep",
            "an abandoned deteriorated house with boarded windows overgrown yard and visible structural damage",
            "an empty vacant lot with no building structure just overgrown grass weeds or bare land"
        ]
        
        text = clip.tokenize(text_prompts).to(device)
        
        with torch.no_grad():
            image_features = model.encode_image(image)
            text_features = model.encode_text(text)
            
            image_features /= image_features.norm(dim=-1, keepdim=True)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        
        prob_occupied = float(similarity[0][0].item() * 100)
        prob_abandoned = float(similarity[0][1].item() * 100)
        prob_vacant_lot = float(similarity[0][2].item() * 100)
        
        scores = [prob_occupied, prob_abandoned, prob_vacant_lot]
        categories = ['occupied', 'abandoned', 'vacant_lot']
        primary_category = categories[scores.index(max(scores))]
        
        if primary_category == 'occupied':
            intervention_score = 0.0
        elif primary_category == 'abandoned':
            intervention_score = prob_abandoned
        elif primary_category == 'vacant_lot':
            intervention_score = prob_vacant_lot * 0.6
        
        scores_dict = {
            'occupied': prob_occupied,
            'abandoned': prob_abandoned,
            'vacant_lot': prob_vacant_lot,
            'primary_category': primary_category,
            'intervention_score': intervention_score
        }
        
        return intervention_score, scores_dict
        
    except Exception as e:
        print(f"Error analyzing image: {e}")
        return None, None

print("✓ CLIP analysis function defined!")

# ============================================================================
# CELL 8: Create Ensemble Predictions
# ============================================================================

def create_ensemble_predictions_with_categories(parcels_df, collection_results_csv):
    """Combine Street View CV analysis with ML model"""
    
    def normalize(series):
        if series.max() == series.min():
            return series * 0
        return (series - series.min()) / (series.max() - series.min())
    
    current_year = pd.Timestamp.now().year
    parcels_df['building_age'] = current_year - parcels_df['YEARBLT']
    parcels_df['building_age'] = parcels_df['building_age'].clip(0, 150)
    
    score_code = normalize(parcels_df['CODE_ENFOR'])
    score_age = normalize(parcels_df['building_age'])
    score_size = 1 - normalize(parcels_df['RESQFT'])
    score_value = 1 - normalize(parcels_df['TOTASSMT'])
    
    parcels_df['ml_score'] = (
        score_code * 0.4 + score_age * 0.3 + 
        score_size * 0.15 + score_value * 0.15
    ) * 100
    
    parcels_df.loc[parcels_df['is_abandoned'] == 1, 'ml_score'] = 100.0
    
    collection_df = pd.read_csv(collection_results_csv)
    
    combined = parcels_df.merge(
        collection_df[['parcel_id', 'filepath', 'has_imagery']],
        left_on='LOCATOR',
        right_on='parcel_id',
        how='left'
    )
    
    print("\n🤖 Running CLIP analysis...")
    
    cv_results = []
    category_counts = {'occupied': 0, 'abandoned': 0, 'vacant_lot': 0, 'no_imagery': 0}
    
    for idx, row in combined.iterrows():
        parcel_data = {
            'parcel_id': row['LOCATOR'],
            'address': row['PROP_ADD'],
            'is_abandoned': row['is_abandoned'],
            'ml_score': row['ml_score'],
            'has_imagery': row.get('has_imagery', False)
        }
        
        if row.get('has_imagery') and pd.notna(row.get('filepath')):
            cv_score, details = analyze_with_clip(row['filepath'])
            
            if cv_score is not None and details is not None:
                parcel_data['cv_score'] = cv_score
                parcel_data['cv_category'] = details['primary_category']
                parcel_data['prob_occupied'] = details['occupied']
                parcel_data['prob_abandoned'] = details['abandoned']
                parcel_data['prob_vacant_lot'] = details['vacant_lot']
                parcel_data['ensemble_score'] = 0.5 * row['ml_score'] + 0.5 * cv_score
                
                category_counts[details['primary_category']] += 1
            else:
                parcel_data.update({
                    'cv_score': None,
                    'cv_category': 'no_imagery',
                    'prob_occupied': None,
                    'prob_abandoned': None,
                    'prob_vacant_lot': None,
                    'ensemble_score': row['ml_score']
                })
                category_counts['no_imagery'] += 1
        else:
            parcel_data.update({
                'cv_score': None,
                'cv_category': 'no_imagery',
                'prob_occupied': None,
                'prob_abandoned': None,
                'prob_vacant_lot': None,
                'ensemble_score': row['ml_score']
            })
            category_counts['no_imagery'] += 1
        
        cv_results.append(parcel_data)
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(combined)} parcels...")
    
    results_df = pd.DataFrame(cv_results)
    results_df.to_csv('../spreadsheets/ferguson_ensemble_scores_with_categories.csv', index=False)
    
    with_imagery = results_df['has_imagery'].sum()
    
    print("\n" + "=" * 70)
    print("✅ ENSEMBLE MODEL COMPLETE")
    print("=" * 70)
    print(f"Total parcels: {len(results_df):,}")
    print(f"Parcels with imagery: {with_imagery:,}")
    print(f"\n📊 CATEGORIES:")
    print(f"  🏠 Occupied: {category_counts['occupied']}")
    print(f"  🏚️  Abandoned: {category_counts['abandoned']}")
    print(f"  🌾 Vacant Lots: {category_counts['vacant_lot']}")
    print(f"  ❓ No imagery: {category_counts['no_imagery']}")
    print(f"\n✓ Saved to: ferguson_ensemble_scores_with_categories.csv")
    print("=" * 70)
    
    return results_df

# RUN ENSEMBLE ANALYSIS
ensemble_results = create_ensemble_predictions_with_categories(
    parcels_df,
    collection_results_csv='streetview_images/collection_results.csv'
)